# 13 · Ataques y daño con el LSTM V4 multianual (2024, 2025, 2026)

Aquí se **abre el sobre**: se califican los 31 detectores entrenados en el notebook 12
sobre los años de prueba, primero con lecturas limpias y después con ataques de
bit flipping. Protocolo: `docs/protocolo_lstm_v4_multianual.md` §7–§8.

**Este libro no entrena ni recalibra nada.** Carga `modelo.keras` y
`umbral_p95_cal2023.txt` de cada estación. Nada de lo que salga aquí puede usarse
para cambiar épocas, umbrales, estaciones o arquitectura: eso sería otro protocolo.

## Cómo usar este libro

1. *Run All*. Si es la primera vez, no hay resultados guardados y verás avisos.
2. Pon `CALCULAR = True` (sección 2) y vuelve a *Run All*. Tarda unos minutos
   (≈4.5 millones de ataques simulados con la cadena real de cifrado). Guarda todo
   y lo anota en la bitácora.
3. Regresa `CALCULAR = False` y guarda: a partir de ahí el libro sólo **lee** los
   resultados, rápido y sin repetir cálculos.

| Flag | Qué hace si es `True` |
|---|---|
| `CALCULAR` | Recalcula la evaluación limpia, la muestra pareada y el barrido, y sobrescribe sus tablas (no toca modelos) |

## 0. Explicación desde cero

### ¿Qué es el ataque?

Cada lectura de ozono viaja como un número entero en binario dentro del mensaje
LoRaWAN cifrado. El atacante no tiene la clave, pero puede **voltear un bit** del
mensaje cifrado y, por cómo funciona el cifrado (AES en modo contador), eso voltea
el mismo bit de la lectura original.

Voltear el bit número *k* (contando desde 0) suma o resta 2^k ppb:

| Bit | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|---|
| Cambio (ppb) | ±1 | ±2 | ±4 | ±8 | ±16 | ±32 | ±64 | ±128 |

**Suma o resta según cómo estaba ese bit**: 87 ppb en binario es 01010111; voltear el
bit 5 (que vale 0) da 119 (+32); en 119 (bit 5 = 1) lo regresa a 87 (−32). El
atacante ciego no sabe cuál de los dos ocurrirá. Aquí se usa la cadena real
(codificar → cifrar → voltear → descifrar → decodificar) y se comprueba contra la
operación XOR en cada lectura.

### ¿Cómo detecta el LSTM?

Para cada hora, el modelo ya calculó (con las vecinas de las 24 h anteriores) cuánto
**esperaba** que midiera la estación. Si lo **recibido** se aleja de lo esperado más
que el umbral de esa estación (p95 de 2023, ~19–33 ppb), salta una **alerta**.
Un ataque de ±1 ppb es imposible de distinguir del error normal; uno de ±128 casi
siempre salta.

### Los cuatro resultados posibles (como una alarma de casa)

| | Hubo ataque | No hubo ataque |
|---|---|---|
| **Sonó la alerta** | **TP** verdadero positivo: lo atrapó | **FP** falso positivo: falsa alarma |
| **No sonó** | **FN** falso negativo: se le escapó | **TN** verdadero negativo: todo bien |

| Métrica | Fórmula | Pregunta que responde |
|---|---|---|
| **Recall** (sensibilidad) | TP / (TP + FN) | De los ataques, ¿cuántos atrapó? |
| **Precisión** | TP / (TP + FP) | Cuando suena, ¿qué tan seguido es un ataque de verdad? |
| **FPR** (tasa de falsas alarmas) | FP / (FP + TN) | De las lecturas limpias, ¿en cuántas sonó por error? |
| **F2** | 5·TP / (5·TP + 4·FN + FP) | Resumen que pesa más no dejar escapar ataques que evitar falsas alarmas |

La precisión depende de cuántos ataques haya: aquí se ataca ~5 % de las lecturas.
Con otra proporción de ataques sería distinta.

### Dos evaluaciones que NO se mezclan

**A. Muestra pareada al 5 %.** En cada estación y año se elige al azar ~5 % de
las lecturas para atacarlas; el resto queda limpio. Se repite con cada bit 0–7 sobre
**las mismas** lecturas elegidas («pareada»), para que la diferencia entre bits se
deba al bit y no al sorteo. De aquí salen recall, precisión, F2 y FPR.

**B. Barrido exhaustivo.** Se ataca **cada** lectura de prueba con **cada** bit,
una a la vez, como si fuera el único ataque del mundo. No mide qué tan seguido
ocurriría un ataque; mide **cuántas oportunidades de causar daño existen** y cuántas
de ellas pasarían sin alerta.

### ¿Qué es «daño»?

No basta con mover el número; daño es **cambiar la decisión**:

| Tipo | Significa |
|---|---|
| `cambio_banda_local` | La lectura atacada cambia de banda NOM-172 (p. ej. de Aceptable a Mala) |
| `cambio_banda_red` | Cambia la banda del **máximo del día de toda la red** de 33 estaciones |
| `falsa_fase1` | El máximo del día era < 155 ppb y con el ataque queda ≥ 155: **excedencia fabricada** |
| `anulada_fase1` | El máximo del día era ≥ 155 y con el ataque queda < 155: **excedencia ocultada** |

Para el máximo de la red sólo se reemplaza la lectura atacada; las demás horas y
estaciones del día se conservan. Ejemplo: bajar 167 → 135 en una estación **no**
oculta nada si otra estación marcó 166 ese día.

Son excedencias del máximo **observado**, no declaraciones oficiales de contingencia.
Una lectura de una estación sin modelo (HGM, XAL, ACO en 2024) puede causar daño,
pero se cuenta como **sin evaluación**: nunca como detectada ni como protegida.

In [1]:
import json, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path('..').resolve()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings('ignore')

from src.detect import multianual as m
from src.detect import entrenamiento_multianual as em
from src.detect import ataques_multianual as at

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)
SAL = RAIZ / 'results/lstm_v4_multianual_v1'
EST = SAL / 'estaciones'
EVT = SAL / 'eventos_dano'          # detalle grande, fuera de git
PLAN = SAL / 'planes_ataque'
IMG = RAIZ / 'docs/img/lstm_v4_multianual_v1'
BITACORA = SAL / 'bitacora_entrenamiento.csv'
for d in (EVT, PLAN, IMG):
    d.mkdir(parents=True, exist_ok=True)

cob = pd.read_csv(SAL / 'cobertura_etapas.csv')
CLASES = cob.drop_duplicates('objetivo').set_index('objetivo')['clasificacion']
print('Modelos disponibles:', len(list(EST.glob('*/modelo.keras'))))

Modelos disponibles: 31


## 1. Datos y predicciones

Se carga la matriz fija de 33 estaciones y las predicciones limpias de cada modelo.
Las predicciones no están en git (opción A): si faltan, se **regeneran cargando el
modelo** (sin entrenar) y se comprueba que el p95 recalculado sea idéntico al guardado.

In [2]:
ctx = em.contexto(RAIZ)
pred = at.cargar_predicciones(EST, ctx)
print('Predicciones:', len(pred), 'filas de', pred.estacion.nunique(), 'estaciones')
print(pred.groupby(pred.timestamp.dt.year).size().rename('lecturas con predicción').to_string())

Predicciones: 533649 filas de 31 estaciones
timestamp
2024    211238
2025    205347
2026    117064


## 2. Calcular o cargar

Con `CALCULAR = False` se leen las tablas guardadas. Con `True` se recalcula todo:
para cada año (2024, 2025, 2026) se arman las lecturas observadas de las 33
estaciones con su contexto diario, se evalúa en limpio, se aplica la muestra pareada
y el barrido exhaustivo.

In [3]:
CALCULAR = False     # ← True para recalcular (unos minutos)

ARCHIVOS = dict(limpias='metricas_limpias.csv', pareadas='metricas_pareadas.csv',
                pareadas_agr='metricas_pareadas_agregadas.csv',
                dano='resumen_dano_mensajes.csv', dias='resumen_dano_dias.csv',
                cobertura='cobertura_prueba.csv')

if CALCULAR:
    t0 = time.time()
    limpias, pareadas, dano, dias, cobertura = [], [], [], [], []
    for anio, etapa in at.ANIOS.items():
        base = at.mensajes_prueba(ctx['valores'], pred, anio)
        # 7.1 limpio, por estación
        for est, g in base[base.evaluable].groupby('estacion'):
            r = em.cargar_objetivo(EST / est)['prueba'][anio]
            limpias.append(dict(anio=anio, estacion=est, clasificacion=CLASES[est], umbral_ppb=g.umbral_ppb.iat[0],
                                **em.metricas_limpias(r, g.umbral_ppb.iat[0])))
        # A. muestra pareada
        sel = at.plan_pareado(base)
        base.loc[sel, ['timestamp', 'estacion', 'original_ppb', 'evaluable']].to_csv(
            PLAN / f'plan_{anio}.csv.gz', index=False)
        pareadas.append(at.evaluar_pareado(base, sel))
        # B. barrido exhaustivo
        c, ev, agr = at.barrido(base)
        dano.append(c)
        dias.append(at.resumen_dias(base, agr))
        ev.to_parquet(EVT / f'eventos_{anio}.parquet', index=False)
        cobertura.append(dict(anio=anio, dias_observados=base.fecha.nunique(),
                              lecturas_observadas=len(base), lecturas_evaluables=int(base.evaluable.sum()),
                              estaciones_con_lecturas=base.estacion.nunique(),
                              estaciones_evaluables=base.loc[base.evaluable, 'estacion'].nunique(),
                              seleccionadas_5pct=int(sel.sum()),
                              dias_max_red_ge155=int(base.groupby('fecha').maximo_original_ppb.first().ge(at.TAU).sum())))
        print(f'{anio}: {len(base):,} lecturas observadas, {int(base.evaluable.sum()):,} evaluables', flush=True)
    R = dict(limpias=pd.DataFrame(limpias), pareadas=pd.concat(pareadas, ignore_index=True),
             dano=pd.concat(dano, ignore_index=True), dias=pd.concat(dias, ignore_index=True),
             cobertura=pd.DataFrame(cobertura))
    R['pareadas_agr'] = at.agregar_pareado(R['pareadas'], CLASES)
    for k, f in ARCHIVOS.items():
        R[k].to_csv(SAL / f, index=False)
    em.registrar(BITACORA, 'evaluar_ataques', 'todos los entrenados', 'calculado',
                 f'años {list(at.ANIOS)}; bits 0-7; tasa {at.TASA}; semilla {at.SEMILLA}',
                 flags=dict(EVALUAR_PRUEBA=True), segundos=time.time() - t0, origen='notebook 13')
    print(f'Listo en {time.time() - t0:.0f} s')
else:
    faltan = [f for f in ARCHIVOS.values() if not (SAL / f).exists()]
    if faltan:
        R = None
        print('Sin resultados guardados todavía; pon CALCULAR = True. Faltan:', faltan)
    else:
        R = {k: pd.read_csv(SAL / f) for k, f in ARCHIVOS.items()}
        print('Resultados cargados de disco.')

Sin resultados guardados todavía; pon CALCULAR = True. Faltan: ['metricas_limpias.csv', 'metricas_pareadas.csv', 'metricas_pareadas_agregadas.csv', 'resumen_dano_mensajes.csv', 'resumen_dano_dias.csv', 'cobertura_prueba.csv']


## 3. Cobertura de la prueba

Cuántas lecturas hubo y cuántas pudo juzgar algún detector. Las no evaluables son de
estaciones sin modelo (HGM, XAL) o sin modelo útil ese año (ACO no tiene datos en
2024). 2026 cubre sólo enero–julio.

In [4]:
if R is not None:
    print(R['cobertura'].to_string(index=False))

## 4. Antes de atacar: lecturas limpias (§7.1)

Igual que la calibración del notebook 12, pero en los años de prueba. Columnas: `mae`
error medio sin signo; `sesgo` (+ = subestima); `sigma` dispersión; `fpr` proporción
de lecturas limpias con alerta (en 2023 era 5 % por construcción; aquí es lo que
realmente pasa); `mae_solo_perfil` el error sin LSTM.

`principal` y `baja_representatividad` se muestran por separado (protocolo §3.2).

In [5]:
if R is not None:
    L = R['limpias']
    resumen = (L.groupby(['anio', 'clasificacion'])
                 .agg(estaciones=('estacion', 'size'), lecturas=('n', 'sum'),
                      mae_mediana=('mae', 'median'), mae_perfil_mediana=('mae_solo_perfil', 'median'),
                      sesgo_mediana=('sesgo', 'median'), fpr_mediana=('fpr', 'median'),
                      fpr_min=('fpr', 'min'), fpr_max=('fpr', 'max')).round(3))
    resumen['falsas_alarmas_total'] = L.groupby(['anio', 'clasificacion']).falsas_alarmas.sum()
    resumen['fpr_micro'] = (resumen.falsas_alarmas_total / resumen.lecturas).round(4)
    print(resumen.to_string())
    print('\nPor estación (2024):')
    print(L[L.anio == 2024].sort_values('fpr', ascending=False)
          [['estacion', 'clasificacion', 'n', 'umbral_ppb', 'mae', 'sesgo', 'sigma', 'fpr', 'mae_solo_perfil']]
          .round(3).to_string(index=False))

## 5. A — Muestra pareada al 5 %: ¿cuántos ataques atrapa?

**Micro** = se suman los TP, FP… de todas las estaciones y se calcula una sola vez
(pesa más quien tiene más lecturas). **Macro** = se calcula por estación y se
promedia (cada estación pesa igual). Si difieren mucho, unas pocas estaciones dominan.

In [6]:
if R is not None:
    A = R['pareadas_agr']
    cols = ['anio', 'bit', 'delta_ppb', 'estaciones', 'tp', 'fn', 'fp', 'tn', 'recall_micro_pct',
            'precision_micro_pct', 'f2_micro_pct', 'fpr_micro_pct', 'recall_macro_pct']
    for grupo in ['principal', 'baja_representatividad']:
        print(f'\n=== {grupo} ===')
        print(A[A.grupo == grupo][cols].round(1).to_string(index=False))

In [7]:
if R is not None:
    A = R['pareadas_agr'][lambda d: d.grupo == 'principal']
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for anio, g in A.groupby('anio'):
        ax[0].plot(g.bit, g.recall_micro_pct, marker='o', label=str(anio) + ('*' if anio == 2026 else ''))
        ax[1].plot(g.bit, g.precision_micro_pct, marker='o', label=str(anio) + ('*' if anio == 2026 else ''))
    ax[0].set_title('Recall: % de ataques atrapados'); ax[1].set_title('Precisión: % de alertas que eran ataque')
    for a in ax:
        a.set_xticks(range(8), [f'{b}\n±{2**b}' for b in range(8)]); a.set_xlabel('bit volteado / ppb')
        a.set_ylim(0, 100); a.grid(alpha=.3); a.legend(title='año (*ene–jul)')
    fig.suptitle('Muestra pareada al 5 %, objetivos principales, agregado micro')
    fig.tight_layout(); fig.savefig(IMG / 'pareada_recall_precision.png', dpi=150); plt.show()

## 6. B — Barrido exhaustivo: oportunidades de daño

Por año, bit y tipo de daño: `oportunidades` (escenarios que causarían ese daño),
`evaluables` (de ellas, con detector), `detectadas`, `no_detectadas` y
`sin_evaluacion` (estación sin modelo). `deteccion_pct` = detectadas / evaluables.
Una oportunidad **no** es una probabilidad: el atacante ciego no sabe qué lectura ni
qué bit causaría el daño.

In [8]:
if R is not None:
    D = R['dano']
    for tipo in at.TIPOS_DANO:
        print(f'\n=== {tipo} ===')
        print(D[D.tipo_dano == tipo][['anio', 'bit', 'delta_ppb', 'oportunidades', 'evaluables', 'detectadas',
                                      'no_detectadas', 'sin_evaluacion', 'deteccion_pct']].round(1).to_string(index=False))

### 6.1 En días

Un día cuenta una vez por bit y tipo, aunque tenga varias oportunidades.
`pct_dias_dano_no_detectado` = días con al menos una oportunidad de daño que pasaría
sin alerta / días observados. Para el **ocultamiento**, lo justo es compararlo con
los días que de verdad tuvieron una excedencia original (`dias_con_excedencia_original`),
no con todos los días.

In [9]:
if R is not None:
    Dd = R['dias']
    cols = ['anio', 'bit', 'n_dias', 'dias_vulnerables_observados', 'dias_vulnerables_evaluables',
            'dias_dano_no_detectado', 'dias_con_dano_sin_evaluacion', 'pct_dias_dano_no_detectado',
            'ataques_daninos_evaluables', 'ataques_daninos_detectados', 'deteccion_dano_pct']
    for tipo in ['falsa_fase1', 'anulada_fase1', 'cambio_banda_red']:
        print(f'\n=== {tipo} ===')
        print(Dd[Dd.tipo_dano == tipo][cols].round(1).to_string(index=False))
    an = Dd[Dd.tipo_dano == 'anulada_fase1'].copy()
    an['pct_ocultables_de_dias_con_excedencia'] = 100 * an.dias_vulnerables_observados / an.dias_con_excedencia_original.where(an.dias_con_excedencia_original > 0)
    an['pct_ocultables_sin_alerta'] = 100 * an.dias_dano_no_detectado / an.dias_con_excedencia_original.where(an.dias_con_excedencia_original > 0)
    print('\nOcultamiento condicionado a días con excedencia original:')
    print(an[['anio', 'bit', 'dias_con_excedencia_original', 'dias_vulnerables_observados', 'dias_dano_no_detectado',
              'pct_ocultables_de_dias_con_excedencia', 'pct_ocultables_sin_alerta']].round(1).to_string(index=False))

In [10]:
if R is not None:
    Dd = R['dias']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
    for ax, tipo, tit in zip(axes, ['falsa_fase1', 'anulada_fase1', 'cambio_banda_red'],
                             ['Excedencia fabricada (≥155)', 'Excedencia ocultada', 'Cambio de banda del máximo de red']):
        for anio, g in Dd[Dd.tipo_dano == tipo].groupby('anio'):
            l, = ax.plot(g.bit, g.dias_vulnerables_evaluables, marker='o', alpha=.35)
            ax.plot(g.bit, g.dias_dano_no_detectado, marker='o', color=l.get_color(),
                    label=f'{anio}{"*" if anio == 2026 else ""}')
        ax.set_title(tit); ax.set_xticks(range(8)); ax.set_xlabel('bit'); ax.grid(alpha=.3)
    axes[0].set_ylabel('días'); axes[0].legend(title='año (*ene–jul)', fontsize=8)
    fig.suptitle('Días con oportunidad de daño (claro) y con daño que pasaría sin alerta (fuerte)')
    fig.tight_layout(); fig.savefig(IMG / 'dano_dias_por_bit.png', dpi=150); plt.show()

## 7. Registro y notas

La ejecución con `CALCULAR = True` queda en `bitacora_entrenamiento.csv`
(acción `evaluar_ataques`). Escribe aquí la interpretación de cada sesión.

#### Plantilla

```
#### AAAA-MM-DD — título
- Qué se hizo:
- Resultados principales (limpio, pareada, barrido, por año):
- Interpretación:
- Decisiones / pendientes:
```

In [11]:
bit = em.leer_bitacora(BITACORA)
print(bit[bit.accion.isin(['evaluar_prueba', 'evaluar_ataques'])][['fecha_hora', 'accion', 'resultado', 'detalle', 'segundos']].to_string(index=False)
      if len(bit) else 'Bitácora vacía')

Empty DataFrame
Columns: [fecha_hora, accion, resultado, detalle, segundos]
Index: []
